In [1]:
import os, sys, platform; print(f"{platform.python_implementation()} {platform.python_version()} | {platform.system()} {platform.release()} | CPU: {platform.processor() or platform.machine()} | logical CPUs: {os.cpu_count()}")

CPython 3.12.13 | Windows 11 | CPU: Intel64 Family 6 Model 151 Stepping 2, GenuineIntel | logical CPUs: 24


In [2]:
from pathlib import Path

INPUT_PATH = Path(r"..\sample\PT\00013 Torso PET AC OSEM")
# INPUT_PATH = Path(r"..\sample\nema\Multiframe\CT\DISCIMG\IMAGES\CT0001")
INPUT_PATH


WindowsPath('../sample/PT/00013 Torso PET AC OSEM')

# Quick VTK timing codelets

Set `INPUT_PATH` to either a DICOM series directory or a single multiframe file.

For a single-file multiframe input such as `CT0001`, `vtk.vtkDICOMImageReader`
on this setup does not reconstruct the full 3D volume. In that case, the `vtk`
timing below is not a like-for-like comparison with `vtkgdcm` or
`dicomsdl.vtk_bridge`, and it may look misleadingly fast because it is reading
only a single slice instead of the full volume.


In [3]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "bindings" / "python" / "dicomsdl").is_dir():
        return cwd
    if cwd.name == "tutorials" and (cwd.parent / "bindings" / "python" / "dicomsdl").is_dir():
        return cwd.parent
    return cwd


def _series_file_names(series_dir: Path) -> list[str]:
    file_names = [
        str(child)
        for child in sorted(series_dir.iterdir())
        if child.is_file() and child.suffix.lower() == ".dcm"
    ]
    if not file_names:
        raise RuntimeError(f"No .dcm files found under {series_dir}")
    return file_names


def _vtk_string_array():
    try:
        from vtkmodules.vtkCommonCore import vtkStringArray
    except ImportError:
        import vtk  # type: ignore
        return vtk.vtkStringArray
    return vtkStringArray


REPO_ROOT = _find_repo_root()
repo_python_dir = REPO_ROOT / "bindings" / "python"
if repo_python_dir.is_dir():
    repo_python_text = str(repo_python_dir)
    if repo_python_text not in sys.path:
        sys.path.insert(0, repo_python_text)

INPUT_PATH = Path(INPUT_PATH)
if not INPUT_PATH.is_absolute():
    INPUT_PATH = (REPO_ROOT / INPUT_PATH).resolve()
if not INPUT_PATH.exists():
    raise FileNotFoundError(INPUT_PATH)

print(f"Repo root: {REPO_ROOT}")
print(f"Input path: {INPUT_PATH}")
print(f"Input kind: {'directory series' if INPUT_PATH.is_dir() else 'single file'}")


Repo root: C:\Lab\workspace\test.git
Input path: C:\Lab\workspace\sample\PT\00013 Torso PET AC OSEM
Input kind: directory series


## `dicomsdl.vtk_bridge.read_series_image_data`

In [4]:
def read_with_dicomsdl_vtk_bridge():
    from dicomsdl.vtk_bridge import read_series_image_data

    return read_series_image_data(INPUT_PATH, to_modality_value=True, copy=False)


%timeit read_with_dicomsdl_vtk_bridge()


105 ms ± 20.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## `vtk.vtkDICOMImageReader`

For a single-file multiframe input, this reader may return only a single slice
instead of the full volume on this setup. Because of that, the `%timeit`
result can look misleadingly fast.


In [5]:
def read_with_vtk():
    import vtk  # type: ignore

    reader = vtk.vtkDICOMImageReader()
    if INPUT_PATH.is_dir():
        reader.SetDirectoryName(str(INPUT_PATH))
    else:
        # Note: on this setup, single-file multiframe inputs are not reconstructed
        # into the full 3D volume by vtkDICOMImageReader.
        reader.SetFileName(str(INPUT_PATH))
    reader.Update()
    return reader.GetOutput()


%timeit read_with_vtk()

408 ms ± 38.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## `vtkgdcm.vtkGDCMImageReader`

In [6]:
from vtkgdcm import vtkGDCMImageReader  # type: ignore

vtkgdcm_file_names = None
if INPUT_PATH.is_dir():
    vtk_string_array = _vtk_string_array()
    vtkgdcm_file_names = vtk_string_array()
    for file_name in _series_file_names(INPUT_PATH):
        vtkgdcm_file_names.InsertNextValue(file_name)


def read_with_vtkgdcm():
    reader = vtkGDCMImageReader()
    if INPUT_PATH.is_dir():
        reader.SetFileNames(vtkgdcm_file_names)
    else:
        reader.SetFileName(str(INPUT_PATH))
    reader.Update()
    return reader.GetOutput()


%timeit read_with_vtkgdcm()

189 ms ± 4.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
